# 集群平台、网络存储与可靠性补充线 · 第 1/8 课：GPU 节点拓扑：NUMA、PCIe、NVLink 与 NIC 亲和性

> 状态：**未开始**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：读取拓扑成本矩阵并为 GPU/NIC 配对，解释错误绑定如何限制多机通信和输入流水。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

`cuda/` 讲 SM/显存，`runtime/` 讲 collective；本课关注节点物理连接、PCIe root、NUMA CPU 和 NIC 的放置。

前置：Linux/网络基础、runtime 补充线、train 分布式章节。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

GPU、NIC、NVMe 挂在 PCIe switch/root complex 下；NVLink/NVSwitch 连接 GPU。进程 CPU affinity、GPU rank 和 NIC 选择共同决定数据路径。

### 数据与控制如何流动

拓扑感知绑定让本地 GPU-NIC 路径尽量少跨 CPU socket/UPI。rank 顺序还会影响 ring 构造。必须以实际 `nvidia-smi topo -m`、NIC NUMA 和 benchmark 为准。

### 正确性条件与常见误区

逻辑 GPU 编号不代表物理邻近；容器可见编号可能重映射。只看 GPU 利用率无法发现 PCIe/NUMA 绕路。

### 性能、成本与工程取舍

最短 GPU-NIC 配对可能让每卡注入带宽更高，但也要考虑 NIC/rail 负载均衡；全部选最近同一 NIC 会形成热点。

## 具体演示

两 GPU、两 NIC 成本矩阵 [[1,5],[4,1]]，配对 G0→N0、G1→N1 总成本 2；交叉配对成本 9。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐最小总成本的一一 GPU-NIC 配对；教学规模使用全排列。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
from itertools import permutations

def best_gpu_nic_pairing(cost):
    n = len(cost)
    if n == 0 or any(len(row) != n for row in cost):
        raise ValueError("cost matrix must be non-empty and square")
    best = None
    for nic_perm in permutations(range(n)):
        # TODO：GPU g 分配到 nic_perm[g]。
        total = ______
        candidate = (total, nic_perm)
        if best is None or candidate < best:
            best = candidate
    return best

assert best_gpu_nic_pairing([[1, 5], [4, 1]]) == (2, (0, 1))


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

为什么 CUDA_VISIBLE_DEVICES=0,1 不能证明两个 rank 在同一 NVLink island？

**你的答案：**


### Q2

GPU 与 NIC 不在同一 PCIe root complex，GPUDirect RDMA 可能受到什么影响？

**你的答案：**


### Q3

为什么“每 GPU 绑定最近 NIC”仍可能不是全局最优？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考资料

- [GPUDirect RDMA](https://docs.nvidia.com/cuda/gpudirect-rdma/index.html)
- [Kubernetes Topology Manager](https://kubernetes.io/docs/tasks/administer-cluster/topology-manager/)
- [NCCL User Guide](https://docs.nvidia.com/deeplearning/nccl/user-guide/index.html)

API 与平台能力会演进；部署前应按目标版本重新核对。